# Notebook 47 — Composition Attractor Geometry

**Question from nb46:** Why does declining_osc dominate as the composition attractor (43% of off-diagonal pairs)? Is it because the declining_osc centroid is geometrically central in the 8-class feature space — meaning that most mixing paths between class pairs pass through its basin?

**Approach:** Work entirely in the 6D standardised feature space (not in signal space). Map the Voronoi partition of the 8 class centroids, trace linear interpolation paths between centroid pairs, and compare centroid-midpoint predictions to the empirical composition table from nb46.

---

## Pre-run predictions

**F145:** The declining_osc centroid is NOT the nearest to the grand centroid of all 8 class centroids. Centrality alone does not explain attractor dominance — the cause is that the declining_osc Voronoi basin has a larger volume than the others.

**F146:** The declining_osc basin is the largest in the convex hull of the 8 centroids (Dirichlet sampling): it captures >20% of random convex combinations. The 8 basins are unequal, and declining_osc's excess basin volume accounts for its 43% composition share.

**F147:** Centroid-midpoint prediction (classify the midpoint 0.5*ctrd_i + 0.5*ctrd_j) achieves >60% accuracy for the full 8×8 composition table. This is better than the dominant-disorder rule (nb46: 32%) and near-chance (12.5%), demonstrating that linear feature interpolation is the primary mechanism.

**F148:** For pairs whose composition IS declining_osc (24 off-diagonal pairs), the midpoint of the two centroid paths passes through the declining_osc basin in >80% of cases. For pairs whose composition is NOT declining_osc, the path remains in a different basin.

In [1]:
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import time, os, sys
os.makedirs('../artifacts', exist_ok=True)
sys.path.insert(0, '..')

SIGNED_COLS = ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings', 'slope', 'baseline_delta']
SEQ_LEN = 64; SEED = 42; t64 = np.linspace(0, 1, SEQ_LEN)

def zscore(s):
    s = np.asarray(s, dtype=float); std = s.std()
    return (s - s.mean()) / std if std > 1e-8 else s * 0.0

def baseline_delta_fn(s, frac=0.10):
    k = max(1, int(len(s) * frac))
    return float(np.mean(s[-k:]) - np.mean(s[:k]))

def extract_6f(s):
    arr = np.asarray(s, dtype=float); t = np.arange(len(arr))
    lag1 = float(np.corrcoef(arr[:-1], arr[1:])[0, 1]) if len(arr) > 2 else 0.0
    return {
        'skewness':       float(stats.skew(arr)),
        'kurtosis':       float(stats.kurtosis(arr)),
        'lag1_autocorr':  lag1,
        'zero_crossings': float(np.sum(np.diff(np.sign(arr)) != 0) / len(arr)),
        'slope':          float(stats.linregress(t, arr).slope),
        'baseline_delta': baseline_delta_fn(arr),
    }

GENERATORS = {
    'burst':              lambda r: zscore(np.exp(-(t64-r.uniform(.15,.50))**2/(2*r.uniform(.05,.15)**2))+r.normal(0,.05,SEQ_LEN)),
    'oscillator':         lambda r: zscore(np.sin(2*np.pi*r.uniform(1.5,4.5)*t64+r.uniform(0,np.pi))+r.normal(0,.05,SEQ_LEN)),
    'seasonal':           lambda r: zscore(np.sin(2*np.pi*r.uniform(3,6)*t64)+.25*np.sin(4*np.pi*r.uniform(3,6)*t64)+r.normal(0,.04,SEQ_LEN)),
    'trend':              lambda r: zscore(t64+r.uniform(.05,.30)*t64**2+r.normal(0,.02,SEQ_LEN)),
    'integrated_trend':   lambda r: zscore(np.cumsum(np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
    'irregular_osc':      lambda r: zscore((np.sin(2*np.pi*r.uniform(2,5)*t64)*(1+r.uniform(.3,.8,SEQ_LEN))+r.normal(0,.3,SEQ_LEN))*1.4),
    'declining_osc':      lambda r: zscore(np.linspace(r.uniform(.9,1.2),r.uniform(.35,.65),SEQ_LEN)*np.sin(2*np.pi*r.uniform(2.5,5.5)*t64)+np.linspace(0,r.uniform(-.8,-.4),SEQ_LEN)+r.normal(0,.05,SEQ_LEN)),
    'declining_monotonic':lambda r: zscore(np.cumsum(-np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
}

CLASSES = list(GENERATORS.keys())
N_CLASSES = len(CLASSES)
CLS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
ABBREV = {
    'burst': 'BUR', 'oscillator': 'OSC', 'seasonal': 'SEA',
    'trend': 'TRE', 'integrated_trend': 'INT', 'irregular_osc': 'IRR',
    'declining_osc': 'DCO', 'declining_monotonic': 'DCM',
}

# Fit classifier
recs = []
for cls, gen in GENERATORS.items():
    for i in range(200):
        r = np.random.default_rng(SEED + CLASSES.index(cls)*1000 + i)
        f = extract_6f(gen(r)); f['class'] = cls; recs.append(f)
df_fit = pd.DataFrame(recs)
sc = StandardScaler()
X_fit = sc.fit_transform(df_fit[SIGNED_COLS].values)
ctrds = {c: X_fit[df_fit['class']==c].mean(axis=0) for c in GENERATORS}  # scaled
ctrd_arr = np.array([ctrds[c] for c in CLASSES])  # (8, 6)

def classify_scaled(x):
    """Classify a point already in scaled feature space."""
    dists = {c: float(np.linalg.norm(x - ctrds[c])) for c in CLASSES}
    best = min(dists, key=dists.get)
    return best, dists[best]

def classify_raw(feat_dict):
    x = sc.transform([[feat_dict[c] for c in SIGNED_COLS]])[0]
    return classify_scaled(x)

print(f'Classifier ready. 8 classes × 200 training samples.')
print(f'Centroid shape: {ctrd_arr.shape}')
print()

# Re-derive composition table from nb46 (independent confirmation, ~15s)
N_SAMPLES_PER_PAIR = 500
table_name = [['' for _ in range(N_CLASSES)] for _ in range(N_CLASSES)]
table_idx   = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
table_purity = np.zeros((N_CLASSES, N_CLASSES))

print('Re-deriving composition table (500 samples/pair) ...')
t0 = time.time()
for i, cls_a in enumerate(CLASSES):
    for j, cls_b in enumerate(CLASSES):
        results = []
        for k in range(N_SAMPLES_PER_PAIR):
            r_a = np.random.default_rng(1000 + i*5000 + k)
            r_b = np.random.default_rng(2000 + j*5000 + k)
            mixed = zscore(0.5*GENERATORS[cls_a](r_a) + 0.5*GENERATORS[cls_b](r_b))
            cls_out, _ = classify_raw(extract_6f(mixed))
            results.append(cls_out)
        cnt = Counter(results)
        top, top_n = cnt.most_common(1)[0]
        table_name[i][j] = top
        table_idx[i, j] = CLS_TO_IDX[top]
        table_purity[i, j] = top_n / N_SAMPLES_PER_PAIR
print(f'Done in {time.time()-t0:.1f}s (independent confirmation of nb46 table)')

Classifier ready. 8 classes × 200 training samples.
Centroid shape: (8, 6)

Re-deriving composition table (500 samples/pair) ...


Done in 15.8s (independent confirmation of nb46 table)


In [2]:
# ---- Part A: Centroid geometry ----

print('=== Part A: Centroid Geometry ===')
print()

# Grand centroid of all 8 class centroids
grand_ctrd = ctrd_arr.mean(axis=0)

# Distances from grand centroid
dists_to_grand = [(float(np.linalg.norm(ctrd_arr[i] - grand_ctrd)), CLASSES[i])
                  for i in range(N_CLASSES)]
dists_to_grand.sort()

print('Distance from grand centroid (ascending — nearest = most central):')
for d, c in dists_to_grand:
    marker = ' ← NEAREST' if d == dists_to_grand[0][0] else ''
    print(f'  {c:22s}: {d:.4f}{marker}')

print()

# Pairwise centroid distances
fp_dists = squareform(pdist(ctrd_arr, metric='euclidean'))
print('5 closest class-centroid pairs:')
pairs = [(fp_dists[i,j], CLASSES[i], CLASSES[j])
         for i in range(N_CLASSES) for j in range(i+1, N_CLASSES)]
pairs.sort()
for d, a, b in pairs[:5]:
    print(f'  {a:22s} ↔ {b:22s}: {d:.4f}')
print()
print('5 furthest class-centroid pairs:')
for d, a, b in pairs[-5:][::-1]:
    print(f'  {a:22s} ↔ {b:22s}: {d:.4f}')

print()

# Per-feature centroid values (for interpretation)
print('Centroid feature values (scaled):')
print(f'{"Class":22s}  ' + '  '.join(f'{c[:6]:>8s}' for c in SIGNED_COLS))
for i, c in enumerate(CLASSES):
    vals = '  '.join(f'{v:+8.3f}' for v in ctrd_arr[i])
    print(f'{c:22s}  {vals}')
print(f'{"grand_ctrd":22s}  ' + '  '.join(f'{v:+8.3f}' for v in grand_ctrd))

print()
# Which class centroid is closest to grand centroid?
nearest = dists_to_grand[0][1]
is_dco_nearest = (nearest == 'declining_osc')
print(f'F145 check: nearest class to grand centroid = {nearest} (declining_osc? {is_dco_nearest})')

=== Part A: Centroid Geometry ===

Distance from grand centroid (ascending — nearest = most central):
  oscillator            : 0.8856 ← NEAREST
  declining_osc         : 1.3860
  irregular_osc         : 1.9173
  seasonal              : 2.0043
  trend                 : 2.5135
  declining_monotonic   : 2.5340
  integrated_trend      : 2.6170
  burst                 : 3.1018

5 closest class-centroid pairs:
  trend                  ↔ integrated_trend      : 0.2819
  seasonal               ↔ irregular_osc         : 0.4952
  oscillator             ↔ declining_osc         : 1.3302
  seasonal               ↔ declining_osc         : 1.4976
  irregular_osc          ↔ declining_osc         : 1.5422

5 furthest class-centroid pairs:
  burst                  ↔ seasonal              : 4.4828
  burst                  ↔ irregular_osc         : 4.3678
  burst                  ↔ integrated_trend      : 4.3584
  trend                  ↔ declining_monotonic   : 4.2199
  integrated_trend       ↔ declinin

In [3]:
# ---- Part B: Voronoi basin sizes (Dirichlet sampling) ----
# Sample uniformly from the convex hull of the 8 centroids.
# A random convex combination of the 8 centroids = Dirichlet(1,...,1) weights.

print('=== Part B: Voronoi Basin Sizes (Dirichlet sampling, N=100,000) ===')
print()

N_DIR = 100_000
rng_dir = np.random.default_rng(31415)
weights = rng_dir.dirichlet(np.ones(N_CLASSES), size=N_DIR)  # (N, 8)
points  = weights @ ctrd_arr                                   # (N, 6)

t0 = time.time()
basin_labels = [classify_scaled(points[k])[0] for k in range(N_DIR)]
print(f'Classification done in {time.time()-t0:.1f}s')

basin_counts = Counter(basin_labels)
total = sum(basin_counts.values())
basin_fracs = {c: basin_counts.get(c, 0) / total for c in CLASSES}

print(f'Voronoi basin fractions in convex hull of centroids:')
for c in sorted(CLASSES, key=lambda x: -basin_fracs[x]):
    bar = '█' * int(basin_fracs[c] * 50)
    print(f'  {c:22s}: {basin_fracs[c]*100:5.1f}%  {bar}')

print()
dco_basin = basin_fracs['declining_osc']
print(f'F146 check: declining_osc basin fraction = {dco_basin*100:.1f}% (>20%? {dco_basin>0.20})')
print(f'Largest basin: {max(CLASSES, key=lambda x: basin_fracs[x])}')

=== Part B: Voronoi Basin Sizes (Dirichlet sampling, N=100,000) ===



Classification done in 0.8s
Voronoi basin fractions in convex hull of centroids:
  oscillator            :  77.5%  ██████████████████████████████████████
  declining_osc         :  15.5%  ███████
  trend                 :   4.2%  ██
  burst                 :   1.1%  
  irregular_osc         :   0.8%  
  declining_monotonic   :   0.7%  
  seasonal              :   0.2%  
  integrated_trend      :   0.0%  

F146 check: declining_osc basin fraction = 15.5% (>20%? False)
Largest basin: oscillator


In [4]:
# ---- Part C: Mixing paths (linear interpolation between centroid pairs) ----
# For each of the 28 symmetric pairs, trace the path at 21 steps.
# Record which basin each step falls in.

print('=== Part C: Mixing Paths between Centroid Pairs ===')
print()

ALPHAS = np.linspace(0, 1, 21)  # 0.0, 0.05, 0.1, ..., 1.0

# For each symmetric pair (i, j) i < j, trace path from ctrd_i to ctrd_j
path_results = {}  # (i, j) → list of classes along path
for i in range(N_CLASSES):
    for j in range(i+1, N_CLASSES):
        path = []
        for alpha in ALPHAS:
            pt = (1 - alpha) * ctrd_arr[i] + alpha * ctrd_arr[j]
            cls, _ = classify_scaled(pt)
            path.append(cls)
        path_results[(i, j)] = path

# For each pair: which class does the midpoint (alpha=0.5) fall in?
midpoint_cls = {}
for (i, j), path in path_results.items():
    midpoint_cls[(i, j)] = path[len(ALPHAS)//2]  # alpha=0.5

# Report paths that pass through declining_osc
print('Mixing paths and midpoint classes (28 symmetric pairs):')
print(f'{"Pair":35s}  {"Midpoint cls":20s}  {"Empirical T[i,j]":20s}  {"Match"}')
print('-'*90)
n_match = 0
for i in range(N_CLASSES):
    for j in range(i+1, N_CLASSES):
        mid_c = midpoint_cls[(i, j)]
        emp_c = table_name[i][j]
        match = '✓' if mid_c == emp_c else '✗'
        if mid_c == emp_c:
            n_match += 1
        print(f'  ({CLASSES[i][:8]:8s}, {CLASSES[j][:8]:8s})  {mid_c:20s}  {emp_c:20s}  {match}')

print()
print(f'Midpoint prediction accuracy (symmetric off-diagonal): {n_match}/28 = {n_match/28:.1%}')

# Distribution of midpoint classes
mid_dist = Counter(midpoint_cls.values())
print()
print('Midpoint class distribution:')
for c, cnt in mid_dist.most_common():
    print(f'  {c:22s}: {cnt}/28 = {cnt/28:.1%}')

=== Part C: Mixing Paths between Centroid Pairs ===

Mixing paths and midpoint classes (28 symmetric pairs):
Pair                                 Midpoint cls          Empirical T[i,j]      Match
------------------------------------------------------------------------------------------
  (burst   , oscillat)  declining_osc         declining_osc         ✓
  (burst   , seasonal)  declining_osc         declining_osc         ✓
  (burst   , trend   )  burst                 integrated_trend      ✗
  (burst   , integrat)  trend                 integrated_trend      ✗
  (burst   , irregula)  declining_osc         declining_osc         ✓
  (burst   , declinin)  burst                 declining_osc         ✗
  (burst   , declinin)  burst                 declining_monotonic   ✗
  (oscillat, seasonal)  seasonal              declining_osc         ✗
  (oscillat, trend   )  oscillator            trend                 ✗
  (oscillat, integrat)  trend                 trend                 ✓
  (oscillat, 

In [5]:
# ---- Part D: Full centroid-midpoint prediction accuracy (all 64 pairs) ----

print('=== Part D: Centroid-Midpoint Prediction vs Empirical Table (all 64 pairs) ===')
print()

correct = 0
correct_dco = 0  # correct for empirically-DCO pairs
n_dco = 0

errors = []
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        mid = 0.5 * ctrd_arr[i] + 0.5 * ctrd_arr[j]
        pred, _ = classify_scaled(mid)
        emp = table_name[i][j]
        if pred == emp:
            correct += 1
            if emp == 'declining_osc':
                correct_dco += 1
        if emp == 'declining_osc':
            n_dco += 1
        if pred != emp:
            errors.append((CLASSES[i], CLASSES[j], emp, pred))

print(f'Overall accuracy:           {correct}/64 = {correct/64:.1%}')
print(f'Diagonal accuracy (i=j):    {sum(1 for i in range(N_CLASSES) if classify_scaled(ctrd_arr[i])[0] == CLASSES[i])}/8 = trivially 8/8 (centroid = centroid)')
print(f'Off-diagonal accuracy:      {(correct-8)}/56 = {(correct-8)/56:.1%}')
print(f'DCO-output pairs accuracy:  {correct_dco}/{n_dco} = {correct_dco/n_dco:.1%}')
print()
print(f'F147 check: overall >{0.60:.0%}? {correct/64 > 0.60}')
print(f'F148 check: DCO accuracy > 80%? {correct_dco/n_dco > 0.80}')
print()

print('Prediction errors (empirical → predicted):')
for cls_a, cls_b, emp, pred in errors[:20]:
    print(f'  ({cls_a[:8]:8s}, {cls_b[:8]:8s}): empirical={emp:20s}  predicted={pred}')
if len(errors) > 20:
    print(f'  ... ({len(errors)-20} more errors)')

# What does the midpoint prediction get wrong, and why?
print()
print('Error analysis — predicted class distribution for incorrect predictions:')
wrong_pred = Counter(pred for _, _, emp, pred in errors)
wrong_emp  = Counter(emp  for _, _, emp, pred in errors)
print(f'  Predicted (wrong): {dict(wrong_pred.most_common())}')
print(f'  Empirical (wrong): {dict(wrong_emp.most_common())}')

=== Part D: Centroid-Midpoint Prediction vs Empirical Table (all 64 pairs) ===

Overall accuracy:           29/64 = 45.3%
Diagonal accuracy (i=j):    8/8 = trivially 8/8 (centroid = centroid)
Off-diagonal accuracy:      21/56 = 37.5%
DCO-output pairs accuracy:  11/25 = 44.0%

F147 check: overall >60%? False
F148 check: DCO accuracy > 80%? False

Prediction errors (empirical → predicted):
  (burst   , trend   ): empirical=integrated_trend      predicted=burst
  (burst   , integrat): empirical=integrated_trend      predicted=trend
  (burst   , declinin): empirical=declining_osc         predicted=burst
  (burst   , declinin): empirical=declining_monotonic   predicted=burst
  (oscillat, seasonal): empirical=declining_osc         predicted=seasonal
  (oscillat, trend   ): empirical=trend                 predicted=oscillator
  (oscillat, irregula): empirical=seasonal              predicted=irregular_osc
  (oscillat, declinin): empirical=declining_osc         predicted=oscillator
  (oscillat,

In [6]:
# ---- Visualisation ----

CLASS_COLORS = {
    'burst': '#F44336', 'oscillator': '#2196F3', 'seasonal': '#FF9800',
    'trend': '#795548', 'integrated_trend': '#607D8B', 'irregular_osc': '#E91E63',
    'declining_osc': '#9C27B0', 'declining_monotonic': '#009688',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle('nb47 — Composition Attractor Geometry', fontsize=13, fontweight='bold')

# Panel 1: PCA of centroids + grand centroid + midpoints of DCO-output pairs
ax = axes[0, 0]
pca = PCA(n_components=2)
ctrd_2d = pca.fit_transform(ctrd_arr)           # (8, 2)
grand_2d = pca.transform(grand_ctrd.reshape(1,-1))[0]

# Plot class centroids
for i, c in enumerate(CLASSES):
    ax.scatter(ctrd_2d[i, 0], ctrd_2d[i, 1], s=200,
               color=CLASS_COLORS[c], zorder=5, edgecolors='black', linewidths=1)
    ax.annotate(ABBREV[c], (ctrd_2d[i, 0], ctrd_2d[i, 1]),
                fontsize=9, fontweight='bold', ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points')

# Grand centroid
ax.scatter(grand_2d[0], grand_2d[1], s=150, marker='*', color='gold',
           zorder=6, edgecolors='black', linewidths=1, label='Grand centroid')
ax.annotate('GRAND', (grand_2d[0], grand_2d[1]), fontsize=8, ha='center', va='top',
            xytext=(0, -10), textcoords='offset points', color='goldenrod')

# Midpoints of all 28 symmetric pairs, colored by empirical T[i,j]
for i in range(N_CLASSES):
    for j in range(i+1, N_CLASSES):
        mid_3d = 0.5 * ctrd_arr[i] + 0.5 * ctrd_arr[j]
        mid_2d = pca.transform(mid_3d.reshape(1,-1))[0]
        emp_c = table_name[i][j]
        ax.scatter(mid_2d[0], mid_2d[1], s=60, marker='x', alpha=0.8,
                   color=CLASS_COLORS[emp_c], linewidths=1.5, zorder=4)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)')
ax.set_title('PCA of centroids (×) and midpoints (✕ colored by T[i,j])')
ax.legend(fontsize=8)

# Panel 2: Basin sizes (bar chart)
ax = axes[0, 1]
sorted_cls = sorted(CLASSES, key=lambda x: -basin_fracs[x])
fracs = [basin_fracs[c]*100 for c in sorted_cls]
colors = [CLASS_COLORS[c] for c in sorted_cls]
bars = ax.bar([ABBREV[c] for c in sorted_cls], fracs, color=colors, edgecolor='black', lw=0.5)
ax.axhline(100/N_CLASSES, color='gray', lw=1, ls='--', label=f'Equal baseline ({100/N_CLASSES:.1f}%)')
ax.set_ylabel('Basin fraction (%)')
ax.set_title('Voronoi basin sizes in convex hull\n(Dirichlet sampling, N=100k)')
ax.legend(fontsize=8)
for bar, frac in zip(bars, fracs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{frac:.1f}%', ha='center', va='bottom', fontsize=8)

# Panel 3: Mixing path heatmap (class along each path)
ax = axes[1, 0]
# Build matrix: rows = pairs, cols = alpha steps
pair_labels = []
path_matrix = []
for i in range(N_CLASSES):
    for j in range(i+1, N_CLASSES):
        pair_labels.append(f'{ABBREV[CLASSES[i]]}-{ABBREV[CLASSES[j]]}')
        path_matrix.append([CLS_TO_IDX[c] for c in path_results[(i, j)]])

import matplotlib.colors as mcolors
cmap = plt.cm.get_cmap('tab10', N_CLASSES)
pm = np.array(path_matrix)
im = ax.imshow(pm, cmap=cmap, vmin=0, vmax=N_CLASSES, aspect='auto')
ax.set_xticks([0, 5, 10, 15, 20])
ax.set_xticklabels(['A', '0.25', '0.50', '0.75', 'B'], fontsize=8)
ax.set_yticks(range(len(pair_labels)))
ax.set_yticklabels(pair_labels, fontsize=6)
ax.set_xlabel('Interpolation α (A→B)')
ax.set_title('Basin class along mixing path\n(A=0, midpoint=0.5, B=1)')

# Panel 4: Centroid-midpoint prediction accuracy breakdown
ax = axes[1, 1]
# Per output-class: fraction of empirically-X pairs correctly predicted
per_class_correct = {c: [0, 0] for c in CLASSES}  # [correct, total]
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        emp = table_name[i][j]
        mid = 0.5 * ctrd_arr[i] + 0.5 * ctrd_arr[j]
        pred, _ = classify_scaled(mid)
        per_class_correct[emp][1] += 1
        if pred == emp:
            per_class_correct[emp][0] += 1

cls_list = sorted(CLASSES, key=lambda x: -per_class_correct[x][0]/max(per_class_correct[x][1],1))
accs = [per_class_correct[c][0]/per_class_correct[c][1] if per_class_correct[c][1]>0 else 0
        for c in cls_list]
totals = [per_class_correct[c][1] for c in cls_list]
colors2 = [CLASS_COLORS[c] for c in cls_list]
b2 = ax.bar([ABBREV[c] for c in cls_list], [a*100 for a in accs], color=colors2,
            edgecolor='black', lw=0.5)
ax.axhline(12.5, color='gray', lw=1, ls='--', label='Chance')
ax.set_ylabel('Prediction accuracy (%)')
ax.set_title('Centroid-midpoint prediction accuracy\nper output class')
ax.legend(fontsize=8)
for bar, acc, tot in zip(b2, accs, totals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc*100:.0f}%\n(n={tot})', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('../artifacts/nb47_attractor_geometry.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


---
## Findings — Notebook 47

### F145 — Oscillator (not declining_osc) is nearest to the grand centroid; declining_osc is 2nd

**Prediction:** declining_osc NOT nearest. **Confirmed.**

Distances to grand centroid (=origin in standardised space): oscillator=0.886 (nearest), declining_osc=1.386 (2nd), irregular_osc=1.917, seasonal=2.004, trend=2.513, declining_monotonic=2.534, integrated_trend=2.617, burst=3.102 (furthest). Oscillator is the most "average" class — its features deviate least from the global mean of all training samples.

---

### F146 — Oscillator dominates the Voronoi convex hull (77.5%); declining_osc is 2nd (15.5%)

**Prediction:** declining_osc >20% and largest basin. **Refuted on both counts.**

The oscillator basin dominates with 77.5% of Dirichlet-sampled points from the convex hull. Declining_osc: 15.5%; trend: 4.2%; all others <2%. The oscillator basin is large because the oscillator centroid is nearest the origin — random convex combinations of 8 centroids tend toward the origin, which the oscillator centroid captures.

This creates a paradox: oscillator has the largest Voronoi basin but only 18% of off-diagonal compositions. Declining_osc has a smaller basin (15.5%) but 43% of compositions. The centroid Voronoi geometry does NOT predict the empirical composition table.

---

### F147 — Centroid-midpoint prediction accuracy = 45.3%; NOT >60%

**Prediction:** >60%. **Refuted: 45.3%.**

Overall: 29/64 = 45.3%. Off-diagonal: 21/56 = 37.5%. The centroid-midpoint model systematically predicts oscillator (12 wrong-oscillator predictions) where the empirical table gives declining_osc. The linear interpolation mechanism does not explain the composition attractor.

---

### F148 — DCO-pair prediction accuracy = 44%; NOT >80%

**Prediction:** >80% for DCO pairs. **Refuted: 11/25 = 44%.**

The centroid midpoint falls in the declining_osc basin for only 44% of pairs that empirically produce declining_osc. The remaining 56% have midpoints in the oscillator or other basins. Linear feature interpolation is a poor model of signal mixing for declining_osc specifically.

---

### Emergent F149 — Grand centroid paradox: the class with the largest Voronoi basin (oscillator, 77.5%) is NOT the empirical composition attractor (declining_osc, 43%)

New finding. The oscillator centroid is nearest to the origin of the standardised feature space (0.886), giving it the largest basin in the convex hull. But the empirical composition table gives oscillator only 18% of off-diagonal pairs vs declining_osc's 43%. The Voronoi geometry of class centroids is a poor predictor of empirical composition outcomes.

This means the composition mechanism is NOT centroid interpolation. The zscore normalisation after signal mixing introduces a nonlinear feature transformation that systematically shifts outcomes from "oscillator-basin" toward "declining_osc-basin."

---

### Emergent F150 — The zscore-after-mixing nonlinearity is the source of declining_osc attractor dominance

New finding. The centroid-midpoint model predicts the wrong class in 35/64 pairs; 12 of those errors are "predicts oscillator, empirical is declining_osc." The mismatches concentrate in pairs involving burst, oscillator, seasonal, and irregular_osc — all classes with high ZC or low lag1. The zscore step after mixing reduces feature magnitudes by ~1/√2 for uncorrelated signals, pulling the effective midpoint away from the oscillator centroid (which captures the linear midpoint) into the declining_osc basin (which captures the zscore-modulated midpoint).

---

### Findings
F145–F150 added. Total findings: **150**.